# Calculate NDVI from STAC Imagery

This notebook computes the **Normalized Difference Vegetation Index (NDVI)** from optical imagery served via STAC. It supports two collection-specific paths:

- **`sentinel2_ard`** — read Red/NIR from a multi-band `cog` asset (B04/B08).
- **`sentinel-2-c1-l2a`** — read Red/NIR from separate `red` and `nir` assets.

Public catalogs can be used without a token; authenticated data require a token in `.env`.

---

## What is NDVI?

NDVI uses the difference between near-infrared (NIR) and red reflectance to indicate vegetation health and density. Values range from **−1** to **+1**:

| Range   | Interpretation        |
|--------|------------------------|
| ≈ 1    | Dense, healthy vegetation |
| ≈ 0    | Bare soil or sparse vegetation |
| ≈ −1   | Water or non-vegetated surfaces |

## Parameters

- **STAC Item & Collection** — Set in the *Set variables* cell. Use the placeholders `{{STAC_ITEM_LINK}}` and `{{STAC_COLLECTION_NAME}}` when running from a template.
- **Token** — Optional. Required only for authenticated catalogs/COGs; store as `token=...` in a `.env` file in this directory.
- **AOI** — Optional GeoJSON geometry. If empty, a 1000×1000 pixel window from the centre of the image is used.

## Workflow

1. Load the STAC item (using token if provided).
2. Resolve red and NIR bands from the selected collection path.
3. Read band data and apply scale/offset when using separate assets.
4. Compute NDVI: **(NIR − Red) / (NIR + Red)**.
5. Visualise the results.

## Import Required Libraries

In [ ]:
%pip install python-dotenv

In [ ]:
import pystac
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
from rasterio.warp import transform_geom
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import json
import warnings
import os
from dotenv import load_dotenv


load_dotenv()

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## Authorisation

For **non-public** datasets, provide a token in a `.env` file in this directory:

```bash
token=your_token_here
```

See [EODH documentation on sensitive data](https://eodatahub.org.uk/docs/documentation/notebooks/sensitive-data/) for details. Public STAC catalogues do not require a token.

In [ ]:
use_auth = False
token = os.getenv("token")
# So rasterio/GDAL send the token when opening COG URLs (avoids 401 on asset hrefs)
if token and use_auth:
    os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {token}"

## Set variables

Set the STAC item URL and collection name for your scene. When launched from a template, the placeholders below may already be filled.

In [ ]:
stac_item_url = "{{STAC_ITEM_LINK}}"
stac_collection_name = "{{STAC_COLLECTION_NAME}}"
aoi_param = """{{AOI}}""".strip()

## Load STAC item

In [ ]:
try:
    stac_io = pystac.StacIO.default()
    stac_io.headers = {"Authorization": f"Bearer {token}"}
    item = pystac.Item.from_file(stac_item_url, stac_io=stac_io)
    print(f"Successfully loaded STAC item: {item.id}")
except Exception as e:
    print(f"Error loading STAC item: {e}")
    raise

## Determine Area of Interest (AOI)

Define the region to process. If you provide a **GeoJSON** geometry (e.g. Polygon or FeatureCollection), the raster is clipped to it. Otherwise, a **1000×1000 pixel** window is taken from the centre of the image.

In [ ]:
DEFAULT_WINDOW_SIZE = 1000
aoi_geometry = None
clip_window = None
use_windowed_read = False

if aoi_param and aoi_param.strip().lower() not in ("", "none", "null"):
    try:
        aoi_data = json.loads(aoi_param)
        if aoi_data.get("type") == "FeatureCollection":
            if aoi_data.get("features") and len(aoi_data["features"]) > 0:
                aoi_geometry = aoi_data["features"][0].get("geometry")
        elif aoi_data.get("type") == "Feature":
            aoi_geometry = aoi_data.get("geometry")
        elif aoi_data.get("type") in ["Polygon", "MultiPolygon", "Point", "LineString"]:
            aoi_geometry = aoi_data
        else:
            raise ValueError(f"Unsupported GeoJSON type: {aoi_data.get('type')}")
        if aoi_geometry and aoi_geometry.get("type"):
            print(f"AOI provided: {aoi_geometry['type']} geometry")
        else:
            raise ValueError("Could not extract geometry from GeoJSON")
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Warning: Could not parse AOI: {e}. Using default window.")
        aoi_geometry = None
else:
    print("No AOI provided, using default 1000×1000 pixel window")

if aoi_geometry is None:
    use_windowed_read = True
    print(
        f"Will extract {DEFAULT_WINDOW_SIZE}×{DEFAULT_WINDOW_SIZE} pixel window from center"
    )

## Access Red and NIR Bands

Band access is collection-specific:

- **`sentinel2_ard`** — use the multi-band `cog` asset and read Red/NIR by band index (B04/B08).
- **`sentinel-2-c1-l2a`** — use the per-band `red` and `nir` assets directly.

In [ ]:
def _asset_extra(asset):
    if isinstance(asset, dict):
        return asset
    return getattr(asset, "extra_fields", None) or getattr(asset, "extra", None) or {}


def _asset_band_entries(asset):
    extra = _asset_extra(asset)
    return extra.get("eo:bands") or extra.get("bands") or []


def get_scale_offset(asset):
    """Read scale and offset from single-band asset metadata."""
    extra = _asset_extra(asset)
    bands = extra.get("bands", []) or extra.get("raster:bands", [])
    if not bands:
        return 1.0, 0.0
    first = bands[0]
    return float(first.get("scale", 1.0)), float(first.get("offset", 0.0))


def _red_nir_indices_from_eo_bands(asset):
    eo_bands = _asset_band_entries(asset)
    if not eo_bands:
        return None, None

    red_idx = None
    nir_idx = None
    for i, b in enumerate(eo_bands, start=1):
        common_name = str(b.get("common_name") or b.get("eo:common_name") or "").lower()
        name = str(b.get("name") or b.get("eo:name") or "").lower()

        if red_idx is None and (common_name == "red" or name == "b04"):
            red_idx = i
        if nir_idx is None and (common_name == "nir" or name == "b08"):
            nir_idx = i

    return red_idx, nir_idx


try:
    red_band_asset = None
    nir_band_asset = None
    cog_asset = None
    red_band_index = 4
    nir_band_index = 8
    red_scale, red_offset = 1.0, 0.0
    nir_scale, nir_offset = 1.0, 0.0

    collection = str(stac_collection_name or "").strip().lower()

    if collection == "sentinel2_ard":
        if "cog" not in item.assets:
            raise ValueError("Expected 'cog' asset for sentinel2_ard.")

        cog_asset = item.assets["cog"]
        red_idx, nir_idx = _red_nir_indices_from_eo_bands(cog_asset)
        red_band_index = red_idx if red_idx is not None else 4
        nir_band_index = nir_idx if nir_idx is not None else 8

        print(
            f"Using sentinel2_ard COG (red=band {red_band_index}, nir=band {nir_band_index})"
        )

    elif collection == "sentinel-2-c1-l2a":
        if "red" not in item.assets or "nir" not in item.assets:
            raise ValueError("Expected 'red' and 'nir' assets for sentinel-2-c1-l2a.")

        red_band_asset = item.assets["red"]
        nir_band_asset = item.assets["nir"]
        red_scale, red_offset = get_scale_offset(red_band_asset)
        nir_scale, nir_offset = get_scale_offset(nir_band_asset)

        print("Using sentinel-2-c1-l2a per-band assets.")
        print(f"Red href: {red_band_asset.href}")
        print(f"NIR href: {nir_band_asset.href}")
        if (
            red_scale != 1.0
            or red_offset != 0.0
            or nir_scale != 1.0
            or nir_offset != 0.0
        ):
            print(
                f"Scale/offset will be applied: R ({red_scale}, {red_offset}), NIR ({nir_scale}, {nir_offset})"
            )

    else:
        raise ValueError(
            "Unsupported collection. Expected 'sentinel2_ard' or 'sentinel-2-c1-l2a'."
        )

except Exception as e:
    print(f"Error accessing bands: {e}")
    print("Available assets:", list(item.assets.keys()))
    raise

## Read Band Data

Read the red and NIR layers from either the `sentinel2_ard` multi-band COG (by band index) or the `sentinel-2-c1-l2a` separate `red`/`nir` assets. Data are clipped to the AOI or to the central 1000×1000 window if no AOI was set. For separate assets, scale and offset from STAC metadata are applied.

In [ ]:
try:
    try:
        _ = aoi_geometry
    except NameError:
        aoi_geometry = None
        clip_window = None
        use_windowed_read = False

    if cog_asset is not None:
        with rasterio.open(cog_asset.href) as src:
            if src.count < max(red_band_index, nir_band_index):
                raise ValueError(
                    f"COG has {src.count} bands; need at least band {max(red_band_index, nir_band_index)}."
                )
            if aoi_geometry is not None:
                # Reproject AOI to raster CRS (GeoJSON is typically WGS84)
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, src.crs, aoi_geometry)
                red_data, red_transform = mask(
                    src, [aoi_in_raster_crs], crop=True, indexes=[red_band_index]
                )
                nir_data, nir_transform = mask(
                    src, [aoi_in_raster_crs], crop=True, indexes=[nir_band_index]
                )
                red_data, nir_data = red_data[0], nir_data[0]
                print(f"Clipped to AOI geometry: {red_data.shape}")
            elif use_windowed_read:
                height, width = src.height, src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    clip_window = None
                    red_data = src.read(red_band_index)
                    nir_data = src.read(nir_band_index)
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                    red_data = src.read(red_band_index, window=clip_window)
                    nir_data = src.read(nir_band_index, window=clip_window)
                print(
                    f"Extracting {DEFAULT_WINDOW_SIZE}×{DEFAULT_WINDOW_SIZE} pixel window from center"
                )
            else:
                red_data = src.read(red_band_index)
                nir_data = src.read(nir_band_index)
            red_crs = src.crs
        red_data = red_data.astype(np.float32)
        nir_data = nir_data.astype(np.float32)
        print(
            f"Read from multi-band COG: band {red_band_index} (red), band {nir_band_index} (NIR)"
        )
    else:
        with rasterio.open(red_band_asset.href) as red_src:
            if aoi_geometry is not None:
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, red_src.crs, aoi_geometry)
                red_data, red_transform = mask(red_src, [aoi_in_raster_crs], crop=True)
                red_data = red_data[0]
            elif use_windowed_read:
                height, width = red_src.height, red_src.width
                if height < DEFAULT_WINDOW_SIZE or width < DEFAULT_WINDOW_SIZE:
                    clip_window = None
                    red_data = red_src.read(1)
                else:
                    row_off = (height - DEFAULT_WINDOW_SIZE) // 2
                    col_off = (width - DEFAULT_WINDOW_SIZE) // 2
                    clip_window = Window(
                        col_off, row_off, DEFAULT_WINDOW_SIZE, DEFAULT_WINDOW_SIZE
                    )
                    red_data = red_src.read(1, window=clip_window)
            else:
                red_data = red_src.read(1)
            red_crs = red_src.crs
        with rasterio.open(nir_band_asset.href) as nir_src:
            if aoi_geometry is not None:
                aoi_crs = "EPSG:4326"
                aoi_in_raster_crs = transform_geom(aoi_crs, nir_src.crs, aoi_geometry)
                nir_data, nir_transform = mask(nir_src, [aoi_in_raster_crs], crop=True)
                nir_data = nir_data[0]
            elif use_windowed_read and clip_window is not None:
                nir_data = nir_src.read(1, window=clip_window)
            else:
                nir_data = nir_src.read(1)
        red_data = red_data.astype(np.float32) * red_scale + red_offset
        nir_data = nir_data.astype(np.float32) * nir_scale + nir_offset

    if red_data.shape != nir_data.shape:
        raise ValueError(
            f"Band shapes do not match: Red {red_data.shape} vs NIR {nir_data.shape}"
        )

    print(f"Red band shape: {red_data.shape}, dtype: {red_data.dtype}")
    print(f"NIR band shape: {nir_data.shape}")
    print(f"CRS: {red_crs}")
    print(f"Band data loaded successfully! Processing {red_data.size:,} pixels")
except Exception as e:
    print(f"Error reading band data: {e}")
    raise

## Calculate NDVI

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

Valid pixels (where the denominator is non-zero) are computed; invalid pixels are set to NaN. Output values are clamped to the range **[−1, 1]**.

In [ ]:
denominator = nir_data + red_data
valid_mask = denominator != 0
ndvi = np.full_like(red_data, np.nan, dtype=np.float32)
ndvi[valid_mask] = (nir_data[valid_mask] - red_data[valid_mask]) / denominator[
    valid_mask
]
ndvi = np.clip(ndvi, -1.0, 1.0)

print("NDVI calculation complete!")
print(
    f"NDVI min: {np.nanmin(ndvi):.4f}, max: {np.nanmax(ndvi):.4f}, mean: {np.nanmean(ndvi):.4f}"
)
print(f"Valid pixels: {np.sum(~np.isnan(ndvi)):,} of {ndvi.size:,}")

## Visualise Results

Red and NIR bands are displayed with a percentile-based stretch (98th percentile) for contrast. The NDVI panel uses a standard colour scale from water (blue) to dense vegetation (green).

In [ ]:
colors = ["#000080", "#0066CC", "#CCCCCC", "#FFFF00", "#00FF00", "#008000"]
cmap = LinearSegmentedColormap.from_list("ndvi", colors, N=256)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmax_red = (
    np.percentile(red_data[~np.isnan(red_data) & (red_data > 0)], 98)
    if np.any(red_data > 0)
    else np.nanmax(red_data)
)
vmax_nir = (
    np.percentile(nir_data[~np.isnan(nir_data) & (nir_data > 0)], 98)
    if np.any(nir_data > 0)
    else np.nanmax(nir_data)
)

axes[0].imshow(red_data, cmap="Reds", vmin=0, vmax=vmax_red)
axes[0].set_title("Red Band", fontsize=14, fontweight="bold")
axes[0].axis("off")
plt.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04, label="Radiance")

axes[1].imshow(nir_data, cmap="YlGn", vmin=0, vmax=vmax_nir)
axes[1].set_title("NIR Band", fontsize=14, fontweight="bold")
axes[1].axis("off")
plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04, label="Radiance")

im3 = axes[2].imshow(ndvi, cmap=cmap, vmin=-1, vmax=1)
axes[2].set_title("NDVI", fontsize=14, fontweight="bold")
axes[2].axis("off")
cbar = plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label="NDVI")
cbar.set_ticks([-1, -0.5, 0, 0.3, 0.6, 1])
cbar.set_ticklabels(["Water", "Bare Soil", "Sparse", "Moderate", "Dense", "Very Dense"])

plt.suptitle(f"NDVI — {item.id}", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("Visualization complete.")

## Summary

- **Supported data**: `sentinel2_ard` (multi-band `cog`) and `sentinel-2-c1-l2a` (`red`/`nir` assets).
- **Auth**: Use a token in `.env` only for authenticated catalogues; public data work without it.
- **Steps**: Load STAC item → resolve red/NIR by collection → read data (with scale/offset for separate assets) → compute NDVI → visualise.
- **Next**: Change `stac_item_url` and `stac_collection_name` for other scenes; optionally set `aoi_param` to a GeoJSON geometry.